In [1]:
import numpy as np
import torch

In [2]:
reviews = [
    "Amazing movie, really enjoyed it.",
    "Great story and excellent acting.",
    "Loved every moment of it.",
    "Very entertaining and enjoyable.",
    "The movie was fantastic.",
    "Brilliant acting and direction.",
    "A fun movie with a great story.",
    "Really impressive and emotional.",
    "One of the best movies I’ve seen.",
    "Highly enjoyable from start to finish.",
    "Very boring and slow.",
    "The story was disappointing.",
    "Poor acting and weak direction.",
    "I didn’t enjoy the movie.",
    "Too predictable and dull.",
    "The movie felt unnecessarily long.",
    "Weak story with poor execution.",
    "Not interesting at all.",
    "A complete waste of time.",
    "The movie was disappointing"
]

labels = [1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0]


In [3]:
MAX_LENGTH = 10

vocabulary = {
    "<PAD>": 0
}


In [4]:
for review in reviews:
    for word in review.split():
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

### Encode all the reviews

In [5]:
def encode(review):
    tokens = [vocabulary[word] for word in review.split()]
    while len(tokens) < MAX_LENGTH:
        tokens.append(0)
    return tokens

In [6]:
encoded_reviews = []
for review in reviews:
    encoded_reviews.append(encode(review))

In [7]:
encoded_reviews

[[1, 2, 3, 4, 5, 0, 0, 0, 0, 0],
 [6, 7, 8, 9, 10, 0, 0, 0, 0, 0],
 [11, 12, 13, 14, 5, 0, 0, 0, 0, 0],
 [15, 16, 8, 17, 0, 0, 0, 0, 0, 0],
 [18, 19, 20, 21, 0, 0, 0, 0, 0, 0],
 [22, 23, 8, 24, 0, 0, 0, 0, 0, 0],
 [25, 26, 19, 27, 28, 29, 30, 0, 0, 0],
 [31, 32, 8, 33, 0, 0, 0, 0, 0, 0],
 [34, 14, 35, 36, 37, 38, 39, 0, 0, 0],
 [40, 41, 42, 43, 44, 45, 0, 0, 0, 0],
 [15, 46, 8, 47, 0, 0, 0, 0, 0, 0],
 [18, 7, 20, 48, 0, 0, 0, 0, 0, 0],
 [49, 23, 8, 50, 24, 0, 0, 0, 0, 0],
 [51, 52, 53, 35, 54, 0, 0, 0, 0, 0],
 [55, 56, 8, 57, 0, 0, 0, 0, 0, 0],
 [18, 19, 58, 59, 60, 0, 0, 0, 0, 0],
 [61, 7, 27, 62, 63, 0, 0, 0, 0, 0],
 [64, 65, 66, 67, 0, 0, 0, 0, 0, 0],
 [25, 68, 69, 14, 70, 0, 0, 0, 0, 0],
 [18, 19, 20, 71, 0, 0, 0, 0, 0, 0]]

### Define a data set

In [8]:
from torch.utils.data import Dataset
# Create the custom dataset
class SentimentDataset(Dataset):
    # Intioalization of Datasets
    def __init__(self):
        #super.__init__(self)
        self.x = torch.tensor(encoded_reviews,dtype=torch.long)
        self.y = torch.tensor(labels,dtype = torch.float32)
    # retuen the length of Dataset
    def __len__(self):
        return len(self.x)
    # return the value at that required index position
    def __getitem__(self,index):
        return self.x[index],self.y[index]

In [9]:
dataset = SentimentDataset()

### Create the DataLoader

In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset=dataset,batch_size=2,shuffle=True)

### Create Model

### Define the model class

In [49]:
# create the subclass of nn.Module to represent the sentiment analysis model
# As till we use the Sequential Class now i go through the own class 
class SetimentAnalysis(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.embeddings = torch.nn.Embedding(
            num_embeddings=len(vocabulary),
            embedding_dim=8
        )
        # Add the RNN Layer
        self.rnn = torch.nn.RNN(
            input_size = 8,
            hidden_size = 16,
            batch_first = True,
            bidirectional = True
        )
        # Add the Linear Layer
        self.linear = torch.nn.Linear(32,1)
        # Apply the Sigmoid activation function
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self,x):
        # convert the Inputs to embeddings 
        x = self.embeddings(x)

        # pass the embeddings to the RNN Layer
        # becoz of batch first hidden state shape : (1,batch_size,hidden_size)
        output , hidden = self.rnn(x)

        # Convert the Shape of the Hidden State
        hidden_cat = torch.cat((hidden[0], hidden[1]), dim=1)

        # pass the reshaped hidden state to the linear layer
        x = self.linear(hidden_cat)

        # get the binary classification result using the sigmoid
        x = self.sigmoid(x)

        return x

### Detect the GPU

In [50]:
device =''
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [51]:
device

'cuda'

In [52]:
model = SetimentAnalysis()

model = model.to(device)

In [53]:
loss_function = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
epochs = 50

### Training Loop

In [55]:
for epoch in range(epochs):
    total_loss = 0
    for x,y in loader:
        x =  x.to(device)
        y = y.to(device).unsqueeze(1) 
        print(y.shape)
        print(y.ndim)
        print(y)
        # pass the input to model and get the Pedictions
        predictions = model(x)
        loss = loss_function(predictions,y)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    print(f"epoch {epoch} , loss : {total_loss}")

torch.Size([2, 1])
2
tensor([[1.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [1.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [1.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [1.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [1.]], device='cuda:0')
epoch 0 , loss : 8.140116393566132
torch.Size([2, 1])
2
tensor([[1.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[1.],
        [1.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor([[0.],
        [0.]], device='cuda:0')
torch.Size([2, 1])
2
tensor

In [18]:
# test the model bu unseen data
def predict(review):
    model.eval()

    # encode the review
    encoded_review = torch.tensor([encode(review)],dtype = torch.long).to(device)

    with torch.no_grad():
        prediction = model(encoded_review)
        print(prediction)
        if prediction>0.5:
            print("Posiitive")
        else:
            print("Negetive")

In [19]:
predict("Brilliant entertaining movie ")

tensor([[[0.5719]],

        [[0.5484]]], device='cuda:0')


RuntimeError: Boolean value of Tensor with more than one value is ambiguous